# CVBoost - Fine-tuning LoRA (partie 3)

Ce notebook tourne sur **Google Colab avec un GPU** (Execution > Modifier le type d'execution > GPU T4).

Etapes :
1. Recuperer le projet et le jeu de donnees anti-fuite (`data/ia/`, cree par `python -m pipeline.dataset_ia`).
2. Charger `google/flan-t5-small` et geler ses poids d'origine.
3. Ajouter un adaptateur LoRA sur les matrices de requete et de valeur de l'attention.
4. Entrainer avec `Seq2SeqTrainer`, evaluer avec SacreBLEU sur le jeu de test.
5. Traitement par lot : le modele lit toutes les offres et produit `offres_ia.csv`.
6. Telecharger l'adaptateur et `offres_ia.csv`, puis les ajouter au depot.

Aucune cle ni aucun secret n'est necessaire : le depot est public.

In [ ]:
# Versions fixees pour que le notebook fonctionne toujours de la meme facon
!pip install -q transformers==4.46.3 peft==0.13.2 datasets==3.1.0 accelerate==1.1.1 sacrebleu==2.4.3

In [ ]:
# Recupere le projet (branche de travail) et se place dedans
BRANCHE = 'claude/capstone-github-setup-g24r7p'
!git clone -q --depth 1 -b {BRANCHE} https://github.com/jason-bit-lang/ball.git
%cd ball

## 1. Jeu de donnees

Les trois fichiers ont deja ete separes sans fuite par `pipeline/dataset_ia.py`.
On reverifie ici qu'aucune empreinte SHA-256 n'est partagee entre deux jeux.

In [ ]:
from datasets import load_dataset

jeux = load_dataset('json', data_files={nom: f'data/ia/{nom}.jsonl' for nom in ['train', 'validation', 'test']})
print(jeux)

empreintes = {nom: set(jeux[nom]['hash']) for nom in jeux}
assert not empreintes['train'] & empreintes['validation'], 'fuite train / validation'
assert not empreintes['train'] & empreintes['test'], 'fuite train / test'
assert not empreintes['validation'] & empreintes['test'], 'fuite validation / test'
print('Aucune fuite entre les jeux')
print(jeux['train'][0]['sortie'])

## 2. Modele de base et adaptateur LoRA

LoRA laisse les poids d'origine **geles** et n'entraine que deux petites matrices ajoutees a cote
des matrices d'attention. Dans T5, les projections de requete et de valeur (appelees `q_proj` et
`v_proj` dans d'autres modeles) s'appellent simplement `q` et `v`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model

MODELE = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(MODELE)
modele_base = AutoModelForSeq2SeqLM.from_pretrained(MODELE)

config_lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,                        # taille des petites matrices ajoutees
    lora_alpha=16,              # force de l'adaptation
    lora_dropout=0.05,
    target_modules=['q', 'v'],  # projections requete et valeur (q_proj / v_proj)
)
modele = get_peft_model(modele_base, config_lora)
modele.print_trainable_parameters()  # moins de 1 % des parametres sont entraines

In [ ]:
ENTREE_MAX, SORTIE_MAX = 512, 64  # longueurs en tokens

def tokeniser(lot):
    exemples = tokenizer(lot['entree'], max_length=ENTREE_MAX, truncation=True)
    exemples['labels'] = tokenizer(text_target=lot['sortie'], max_length=SORTIE_MAX, truncation=True)['input_ids']
    return exemples

jeux_tokens = jeux.map(tokeniser, batched=True, remove_columns=jeux['train'].column_names)

## 3. Entrainement avec Seq2SeqTrainer et score SacreBLEU

In [ ]:
import numpy as np
import sacrebleu
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

def decoder(ids):
    ids = np.where(ids != -100, ids, tokenizer.pad_token_id)  # -100 = case vide a ignorer
    return [t.strip() for t in tokenizer.batch_decode(ids, skip_special_tokens=True)]

def calculer_bleu(prediction):
    predictions, references = decoder(prediction.predictions), decoder(prediction.label_ids)
    return {'bleu': sacrebleu.corpus_bleu(predictions, [references]).score}

arguments = Seq2SeqTrainingArguments(
    output_dir='resultats',
    num_train_epochs=20,               # peu d'exemples : on les revoit plusieurs fois
    learning_rate=1e-3,                # LoRA supporte un pas plus grand qu'un fine-tuning complet
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,       # garde l'epoque qui a le meilleur BLEU en validation
    metric_for_best_model='bleu',
    predict_with_generate=True,
    generation_max_length=SORTIE_MAX,
    fp16=False,                        # T5 est instable en demi-precision
    logging_steps=10,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=modele,
    args=arguments,
    train_dataset=jeux_tokens['train'],
    eval_dataset=jeux_tokens['validation'],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=modele),
    tokenizer=tokenizer,
    compute_metrics=calculer_bleu,
)
trainer.train()

In [ ]:
# Score final sur le jeu de test, jamais vu pendant l'entrainement
score_lora = trainer.evaluate(jeux_tokens['test'], metric_key_prefix='test')['test_bleu']

# Meme mesure avec le modele d'origine (adaptateur desactive) pour voir ce que LoRA apporte
with modele.disable_adapter():
    score_base = trainer.evaluate(jeux_tokens['test'], metric_key_prefix='test')['test_bleu']

print(f'SacreBLEU test - modele de base : {score_base:.1f}')
print(f'SacreBLEU test - avec LoRA      : {score_lora:.1f}')

In [ ]:
# Quelques exemples du jeu de test : sortie attendue / sortie du modele
def generer(textes, taille_lot=16):
    modele.eval()
    sorties = []
    for i in range(0, len(textes), taille_lot):
        entrees = tokenizer(textes[i:i + taille_lot], max_length=ENTREE_MAX, truncation=True,
                            padding=True, return_tensors='pt').to(modele.device)
        with torch.no_grad():
            ids = modele.generate(**entrees, max_new_tokens=SORTIE_MAX)
        sorties += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return sorties

for exemple, sortie in zip(jeux['test'].select(range(3)), generer(jeux['test']['entree'][:3])):
    print('Attendu :', exemple['sortie'])
    print('Modele  :', sortie, '\n')

## 4. Traitement par lot de toutes les offres

Le modele lit chaque offre scrapee et produit l'intitule et les competences.
Le site lit ensuite ce fichier : aucun modele ne tourne sur Render.

In [ ]:
import pandas as pd

PREFIXE, DESCRIPTION_MAX = 'Extraire le poste et les competences : ', 2000  # memes reglages que pipeline/dataset_ia.py
offres = pd.read_csv('data/offres_propres.csv').dropna(subset=['url', 'description'])
sorties = generer([PREFIXE + d[:DESCRIPTION_MAX] for d in offres['description']])

def lire_sortie(sortie):
    """'Poste : X | Competences : a, b' -> ('X', 'a, b')"""
    poste, _, competences = sortie.partition('| Competences :')
    return poste.replace('Poste :', '').strip(), competences.strip()

offres_ia = pd.DataFrame({'url': offres['url']})
offres_ia['titre_ia'], offres_ia['competences_ia'] = zip(*map(lire_sortie, sorties))
offres_ia.to_csv('offres_ia.csv', index=False)
offres_ia.head()

## 5. Telechargement

Deux fichiers a ajouter au depot :
- `offres_ia.csv` -> `data/offres_ia.csv`
- `lora_cvboost.zip` -> a decompresser dans `modele/lora_cvboost/` (quelques Mo seulement : seul l'adaptateur est garde)

In [ ]:
modele.save_pretrained('lora_cvboost')   # enregistre uniquement l'adaptateur LoRA
tokenizer.save_pretrained('lora_cvboost')
!zip -q -r lora_cvboost.zip lora_cvboost
!du -sh lora_cvboost

from google.colab import files
files.download('offres_ia.csv')
files.download('lora_cvboost.zip')